In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=42)

In [3]:
df = pd.read_excel(
    "../../../data/hospital_billing.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "isCancelled": "string",
        "isClosed": "string",
        "caseType": "string",
        "speciality": "string",
        "blocked": "string",
        "flagD": "string",
        "flagB": "string",
        "flagA": "string",
        "state": "string",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,blocked,caseType,concept:name,flagA,flagB,flagD,isCancelled,isClosed,lifecycle:transition,speciality,state,time_delta
0,A,2012-12-16 19:33:10,False,A,NEW,False,False,True,False,True,complete,A,In progress,0.0
1,A,2013-12-15 19:00:37,NA,NA,FIN,NA,NA,NA,NA,NA,complete,NA,Closed,31447648.0
2,A,2013-12-16 03:53:38,NA,NA,RELEASE,NA,NA,NA,NA,NA,complete,NA,Released,31981.0
3,A,2013-12-17 12:56:29,NA,NA,CODE OK,NA,NA,NA,NA,NA,complete,NA,NA,118971.0
4,A,2013-12-19 03:44:31,NA,NA,BILLED,NA,NA,NA,NA,NA,complete,NA,Billed,139682.0
5,AA,2012-12-26 08:50:18,False,B,NEW,False,False,True,False,True,complete,L,In progress,0.0
6,AA,2012-12-26 08:50:59,NA,NA,CHANGE DIAGN,NA,NA,NA,NA,NA,complete,NA,In progress,41.0
7,AA,2013-02-14 21:06:33,NA,NA,FIN,NA,NA,NA,NA,NA,complete,NA,Closed,4364134.0
8,AA,2013-02-14 22:12:10,NA,NA,RELEASE,NA,NA,NA,NA,NA,complete,NA,Released,3937.0
9,AA,2013-02-18 01:44:10,NA,NA,CODE OK,NA,NA,NA,NA,NA,complete,NA,NA,271920.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['blocked', 'caseType', 'concept:name', 'flagA', 'flagB', 'flagD', 'isCancelled', 'isClosed', 'lifecycle:transition', 'speciality', 'state', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
isCancelled                    categorical    event    yes    ['False', 'True']                        N/A        data_derived        
isClosed                       categorical    event    yes    ['False', 'True']                        N/A        data_derived        
caseType                       categorical    event    yes    ['A', 'B', 'C', ...]                     N/A     

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load()

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'CODE OK', 'RELEASE'}]

In [16]:
engine.branching_sets

[{'BILLED',
  'CHANGE DIAGN',
  'CHANGE END',
  'CODE ERROR',
  'CODE NOK',
  'CODE OK',
  'DELETE',
  'FIN',
  'JOIN-PAT',
  'MANUAL',
  'REJECT',
  'RELEASE',
  'REOPEN',
  'SET STATUS',
  'STORNO',
  'ZDBC_BEHAN'},
 {'CHANGE DIAGN',
  'CHANGE END',
  'CODE ERROR',
  'CODE NOK',
  'CODE OK',
  'FIN',
  'MANUAL',
  'REJECT',
  'RELEASE',
  'REOPEN',
  'STORNO'},
 {'CHANGE DIAGN',
  'CHANGE END',
  'CODE ERROR',
  'FIN',
  'MANUAL',
  'REJECT',
  'REOPEN',
  'STORNO'},
 {'CHANGE DIAGN', 'CHANGE END', 'CODE ERROR', 'REJECT', 'REOPEN', 'STORNO'}]

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/hospital_billing-cf_generated_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/420 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,YAWA,3,1,0,0.068830,0.137660,0.000000,0.05,0.555556,...,1.632563,0.555556,0.068830,0.000000,0.137660,0.05,0.958177,0.958177,1.0,1.0
1,0,VSOB,4,1,0,0.050149,0.044742,0.055556,0.10,0.818182,...,1.946908,0.818182,0.050149,0.055556,0.044742,0.10,0.978577,0.801513,1.0,1.0
2,0,SNUD,5,1,0,0.040779,0.081557,0.000000,0.05,0.230769,...,1.170362,0.230769,0.040779,0.000000,0.081557,0.05,0.848815,0.848815,1.0,1.0
3,0,SVAE,6,1,2,0.083380,0.000094,0.166667,0.25,0.266667,...,0.600047,0.266667,0.083380,0.166667,0.000094,0.25,0.000000,0.000000,0.0,0.0
4,0,YEVB,7,1,0,0.055671,0.000232,0.111111,0.15,0.470588,...,0.676260,0.470588,0.055671,0.111111,0.000232,0.15,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365,41,RPR,11,1,4,0.110402,0.220804,0.000000,0.09,0.440000,...,1.614740,0.440000,0.110402,0.000000,0.220804,0.09,0.974338,0.944758,1.0,1.0
366,41,UHJB,11,1,4,0.050092,0.100185,0.000000,0.09,0.520000,...,1.630813,0.520000,0.050092,0.000000,0.100185,0.09,0.970720,0.000000,1.0,0.0
367,41,GNDB,11,1,4,0.050085,0.100170,0.000000,0.09,0.520000,...,1.631332,0.520000,0.050085,0.000000,0.100170,0.09,0.971247,0.000000,1.0,0.0
368,41,CDMA,12,1,4,0.048123,0.096246,0.000000,0.09,0.481481,...,1.589976,0.481481,0.048123,0.000000,0.096246,0.09,0.970372,0.000000,1.0,0.0


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/420 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,YAWA,3,1,0,0.348591,0.197182,0.500000,0.50,0.333333,...,2.110085,0.333333,0.348591,0.500000,0.197182,0.50,0.928161,0.000000,1.0,0.0
1,0,VSOB,4,1,0,0.488371,0.587853,0.388889,0.45,0.181818,...,1.120189,0.181818,0.488371,0.388889,0.587853,0.45,0.000000,0.000000,0.0,0.0
2,0,SNUD,5,1,0,0.457537,0.526185,0.388889,0.45,0.307692,...,1.215229,0.307692,0.457537,0.388889,0.526185,0.45,0.000000,0.000000,0.0,0.0
3,0,SVAE,6,1,2,0.444454,0.555575,0.333333,0.40,0.266667,...,1.581850,0.266667,0.444454,0.333333,0.555575,0.40,0.470729,0.470729,0.0,0.0
4,0,YEVB,7,1,0,0.370377,0.462977,0.277778,0.35,0.235294,...,1.381343,0.235294,0.370377,0.277778,0.462977,0.35,0.425672,0.425672,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365,41,RPR,11,1,4,0.412784,0.370012,0.455556,0.50,0.920000,...,2.778538,0.920000,0.412784,0.455556,0.370012,0.50,0.945754,0.477814,1.0,0.0
366,41,UHJB,11,1,4,0.051693,0.103387,0.000000,0.09,0.520000,...,1.632410,0.520000,0.051693,0.000000,0.103387,0.09,0.970717,0.000000,1.0,0.0
367,41,GNDB,11,1,4,0.056566,0.113132,0.000000,0.09,0.520000,...,1.637764,0.520000,0.056566,0.000000,0.113132,0.09,0.971198,0.000000,1.0,0.0
368,41,CDMA,12,1,4,0.049460,0.098919,0.000000,0.09,0.481481,...,1.591299,0.481481,0.049460,0.000000,0.098919,0.09,0.970357,0.000000,1.0,0.0


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()